# Podcast AI Intel Synthesizer

### Automated Speech-to-Insights Engine for *Latent Space (The AI Engineer Podcast)*

In [1]:
# Install Core Dependencies

!pip install -q --upgrade transformers accelerate bitsandbytes feedparser pydub datasets requests
!apt-get install -y -qq ffmpeg


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
# Check GPU Hardware and Available VRAM

import torch
print("=" * 50)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f"Total GPU VRAM: {vram_gb} GB")
else:
    print("WARNING: No GPU detected! Go to: Runtime -> Change runtime type -> T4 GPU")
print("=" * 50)


CUDA Available: True
GPU Device Name: Tesla T4
Total GPU VRAM: 15.64 GB


In [3]:
# Core Imports and Hugging Face Authentication

import os
import time
import requests
import feedparser
from IPython.display import Markdown, display
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextStreamer,
    BitsAndBytesConfig,
    pipeline
)

# Fetch HF token from Colab Secrets or Environment

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=True)
    print("Hugging Face Hub authenticated successfully!")
else:
    print(" No HF_TOKEN found in Colab Secrets. If you haven't added it, add 'HF_TOKEN' under the Secrets (🔑) tab.")


Hugging Face Hub authenticated successfully!


In [4]:
# Ingest Latest Episode from Latent Space Podcast RSS

LATENT_SPACE_RSS_URL = "https://api.substack.com/feed/podcast/1084089.rss"

def fetch_latest_podcast_episode(rss_url: str, output_path: str = "latest_episode.mp3"):
    print(f" Connecting to RSS Feed: {rss_url}...")
    feed = feedparser.parse(rss_url)

    # Safety check: Did the feed return any episodes?

    if not feed.entries:
        raise ValueError("Could not find any episodes in this RSS feed. Please check the URL.")

    # 1. Grab the latest episode (Index 0 is always the newest release)

    latest = feed.entries[0]
    title = latest.get("title", "Untitled Episode")
    pub_date = latest.get("published", "Unknown Date")
    summary = latest.get("summary", "")

    print(f" Found Latest Episode: {title}")
    print(f" Published: {pub_date}")

    # 2. Extract the direct MP3 link from the feed's 'enclosures'

    audio_url = None
    if "enclosures" in latest and len(latest.enclosures) > 0:
        audio_url = latest.enclosures[0].get("href")
    elif "links" in latest:
        for link in latest.links:
            if link.get("type", "").startswith("audio/"):
                audio_url = link.get("href")
                break

    if not audio_url:
        raise ValueError(f"No direct audio link found for: {title}")

    # 3. Stream download the MP3 file safely to disk

    print(f" Downloading audio file...")
    response = requests.get(audio_url, stream=True)
    response.raise_for_status()

    with open(output_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):  # 1 MB at a time
            if chunk:
                f.write(chunk)

    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f" Download complete: {output_path} ({file_size_mb:.2f} MB)")

    return {
        "title": title,
        "published": pub_date,
        "summary": summary,
        "audio_url": audio_url,
        "audio_path": output_path
    }

# Run the function to fetch the episode

episode_info = fetch_latest_podcast_episode(LATENT_SPACE_RSS_URL)


 Connecting to RSS Feed: https://api.substack.com/feed/podcast/1084089.rss...
 Found Latest Episode: Claude Code’s Next Era — Thariq Shihipar, Anthropic
 Published: Tue, 29 Sep 2026 01:48:18 GMT
 Download complete: latest_episode.mp3 (84.67 MB)


In [5]:
#  Audio Preprocessing & Fast Dev Slice Toggle
from pydub import AudioSegment

# -------------------------------------------------------------
# TOGGLE SWITCH:
# True  = Slice first 10 minutes (transcribes in ~35 seconds)
# False = Process the full 60-90 minute episode
# -------------------------------------------------------------
FAST_TEST_MODE = True
TEST_MINUTES = 10

if FAST_TEST_MODE:
    print(f" Fast Dev Mode Enabled: Slicing the first {TEST_MINUTES} minutes...")
    # 1. Load the downloaded MP3 file
    audio = AudioSegment.from_file(episode_info["audio_path"])

    # 2. Convert minutes to milliseconds (pydub calculates time in milliseconds!)
    duration_ms = TEST_MINUTES * 60 * 1000

    # 3. Slice from the beginning (0) up to 10 minutes
    sliced_audio = audio[:duration_ms]

    # 4. Save the slice as a new lightweight MP3 file
    ACTIVE_AUDIO_FILE = "episode_test_slice.mp3"
    sliced_audio.export(ACTIVE_AUDIO_FILE, format="mp3")
    print(f" Created {TEST_MINUTES}-minute slice: {ACTIVE_AUDIO_FILE}")
else:
    ACTIVE_AUDIO_FILE = episode_info["audio_path"]
    print(f" Full episode active: {ACTIVE_AUDIO_FILE}")


 Fast Dev Mode Enabled: Slicing the first 10 minutes...


/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


 Created 10-minute slice: episode_test_slice.mp3


In [6]:
#  Speech-to-Text Pipeline with Whisper Large v3 Turbo
print("Initializing Whisper Large v3 Turbo ASR pipeline...")

whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    torch_dtype=torch.float16,
    device="cuda:0" if torch.cuda.is_available() else "cpu",
    chunk_length_s=30,      # Process in 30-second audio windows
    batch_size=8,           # Process 8 chunks at once in parallel on GPU
    return_timestamps=True
)

print(f" Transcribing: {ACTIVE_AUDIO_FILE}...")
start_time = time.time()

# Run the speech-to-text pipeline
transcription_result = whisper_pipe(ACTIVE_AUDIO_FILE)
transcription = transcription_result["text"]

elapsed = round(time.time() - start_time, 2)
word_count = len(transcription.split())

print(f" Transcription complete in {elapsed} seconds!")
print(f" Total words transcribed: {word_count}")
print("\n--- Transcription Preview (First 350 characters) ---")
print(transcription[:350] + "...")


Initializing Whisper Large v3 Turbo ASR pipeline...


config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.62GB            

model.safetensors: downloading bytes:           |  0.00B            

...

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


 Transcribing: episode_test_slice.mp3...


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

 Transcription complete in 46.43 seconds!
 Total words transcribed: 1850

--- Transcription Preview (First 350 characters) ---
 We're here in the studio with our friend Tarek from Enthropic, and I guess generally the Cloud Code, there's so much sort of merging of boundaries, and you've been so on top of everything since you joined Enthropic. You have been early to Cloud Code itself, but then also, and you've told that story in other podcasts, and you've also been talking a...


In [7]:
# Load Quantized LLM (Meta Llama 3.2 3B Instruct)
LLAMA_MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

# Configure 4-bit NF4 Quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

print(f"Loading tokenizer & 4-bit model: {LLAMA_MODEL_ID}...")

# 1. Load Tokenizer (Converts words into number IDs)
tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# 2. Load the Model Weights onto GPU
model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL_ID,
    device_map="auto",
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16
)

print(f"{LLAMA_MODEL_ID} loaded successfully into GPU VRAM!")


Loading tokenizer & 4-bit model: meta-llama/Llama-3.2-3B-Instruct...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

meta-llama/Llama-3.2-3B-Instruct loaded successfully into GPU VRAM!


In [8]:
# Domain-Specific System & User Prompt Engineering

# 1. System Prompt: Sets the "Persona" and "Rules" for the AI
system_message = """You are an elite AI systems researcher and executive tech editor specializing in cutting-edge machine learning podcasts (such as Latent Space).

Your mission is to distill dense, highly technical conversations into an insightful, executive-level technical briefing.

Core Guidelines:
- Highlight concrete architectural details, engineering trade-offs, model names, and benchmark insights.
- Distinguish between established engineering facts and controversial/forward-looking speculations.
- Attribute viewpoints specifically to speakers (e.g. Swyx, Alessio, or the invited guest).
- Preserve intellectual depth; avoid vague marketing jargon or generic summaries.
- Return clean GitHub-flavored Markdown without wrapping the entire response in code blocks.
"""

# 2. User Prompt: Injects the real podcast episode info and transcript
user_prompt = f"""Below is the speech-to-text transcript from an episode of the Latent Space AI Engineer podcast.

Episode Title: {episode_info['title']}
Published Date: {episode_info['published']}

Please analyze the transcript and generate an in-depth Technical Intel Briefing organized into the following sections:

#  Executive Summary & Core Narrative
- **Episode Thesis:** 2-3 sentences capturing the primary focus of the discussion.
- **Key Problem Addressed:** What challenge in the current AI/ML landscape is discussed?

# Technical Deep Dives & Architectural Trade-offs
(Identify the 3-4 major technical concepts or engineering systems discussed. For each, explain the rationale, trade-offs, and design patterns.)

#  Contrarian Takes & Bold Claims
- Highlight viewpoints that challenge current consensus in the LLM/AI engineering space.

# Predictions, Milestones & Timelines
- Specific timelines, upcoming model releases, or industry shifts predicted by the guests.

#  High-Signal Quotes
- 2-3 verbatim or near-verbatim quotes that capture the essence of the conversation with brief context.

#  Actionable Takeaways for AI Engineers
- Bulleted list of practical engineering patterns, tools, or techniques listeners can implement today.

---
Transcript:
{transcription}
"""

# 3. Package as standard chat messages list
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
]

print(" Prompts prepared and formatted for Llama chat template!")
print(f" Prompt contains {len(user_prompt.split())} words (including transcript).")


 Prompts prepared and formatted for Llama chat template!
 Prompt contains 2033 words (including transcript).


In [9]:
# Run Generation with Real-Time Streaming and Clean Token Slicing

# 1. Format prompt using Llama 3's official chat template
prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# 2. Convert text to token numbers and move safely to GPU
inputs = tokenizer(prompt_text, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}
input_token_count = inputs["input_ids"].shape[1]

# 3. Enable real-time word-by-word streaming to screen
streamer = TextStreamer(tokenizer, skip_prompt=True)

print(f" Generating Technical Briefing ({input_token_count} input tokens)...")
print("=" * 65 + "\n")

# 4. Generate the response
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2500,    # Max length of the briefing
        temperature=0.35,       # Low temperature = high factual accuracy, less creative fluff
        top_p=0.9,
        do_sample=True,
        streamer=streamer
    )

print("\n" + "=" * 65)
print(" Generation complete!")

# 5. Clean Token Slicing: Strip out the input prompt and decode clean Markdown
generated_tokens = outputs[0][input_token_count:]
briefing_markdown = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


 Generating Technical Briefing (2609 input tokens)...



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


**Executive Summary & Core Narrative**

Thariq Shihipar, Anthropic's engineer, discusses the evolution of Cloud Code and its next era, highlighting the challenges of scaling and the importance of human-agent interaction. He emphasizes the need for a more efficient and intuitive interface, particularly in harnessing the capabilities of the Ask User Question tool and artifacts. Thariq also touches on the distinction between local and cloud-based systems, suggesting a future where local clouds can communicate with each other seamlessly.

**Key Problem Addressed**

The primary challenge discussed is the need for a more efficient and intuitive interface to harness the capabilities of Cloud Code, particularly in harnessing the Ask User Question tool and artifacts. Thariq highlights the difficulties in scaling and the importance of human-agent interaction in this context.

**Technical Deep Dives & Architectural Trade-offs**

1. **Ask User Question Tool**: Thariq discusses the innovative natur

In [10]:
# Render Markdown in Notebook & Auto-Save to Disk
import time
from IPython.display import Markdown, display

# 1. Render rich, beautifully styled Markdown inside the notebook
display(Markdown(briefing_markdown))

# 2. Sanitize episode title to remove illegal filename characters (like ':', '/', or '?')
safe_title = "".join(c for c in episode_info["title"] if c.isalnum() or c in (" ", "_", "-")).rstrip()
output_md_filename = f"Latent_Space_Briefing_{safe_title[:35].replace(' ', '_')}.md"

# 3. Save the full briefing with metadata header to a .md file
with open(output_md_filename, "w", encoding="utf-8") as f:
    f.write(f"# {episode_info['title']}\n\n")
    f.write(f"**Podcast:** Latent Space (The AI Engineer Podcast)\n")
    f.write(f"**Published:** {episode_info['published']}\n")
    f.write(f"**Generated:** {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write("---\n\n")
    f.write(briefing_markdown)

print(f"\n✓ Executive briefing saved to disk: {output_md_filename}")


**Executive Summary & Core Narrative**

Thariq Shihipar, Anthropic's engineer, discusses the evolution of Cloud Code and its next era, highlighting the challenges of scaling and the importance of human-agent interaction. He emphasizes the need for a more efficient and intuitive interface, particularly in harnessing the capabilities of the Ask User Question tool and artifacts. Thariq also touches on the distinction between local and cloud-based systems, suggesting a future where local clouds can communicate with each other seamlessly.

**Key Problem Addressed**

The primary challenge discussed is the need for a more efficient and intuitive interface to harness the capabilities of Cloud Code, particularly in harnessing the Ask User Question tool and artifacts. Thariq highlights the difficulties in scaling and the importance of human-agent interaction in this context.

**Technical Deep Dives & Architectural Trade-offs**

1. **Ask User Question Tool**: Thariq discusses the innovative nature of this tool, which enables the model to elicit requirements from users. He notes that it's an emergent behavior that he wanted to see if the models could do. The tool splits the user's intent into two categories: those who want the agent to do the work and those who need clarification on their requirements.
2. **Artifacts**: Thariq explains that artifacts are a way to store and manage persistent data, which can be accessed by multiple Clouds. He emphasizes the importance of dashboard artifacts, which can store Kanban data and enable multiple Clouds to access and communicate with each other.
3. **Cloud Mods**: Thariq mentions that Cloud Mods have come a long way since their inception, with features like Ask User Question and artifacts. He notes that the next step is to create a more on-the-fly interface for harnessing these capabilities.
4. **Local and Cloud-based Systems**: Thariq discusses the distinction between local and cloud-based systems, suggesting a future where local clouds can communicate with each other seamlessly. He envisions a system where you can message a local cloud, which starts a session locally and executes, rather than relying on cloud-to-cloud communication.

**Contrarian Takes & Bold Claims**

* Thariq's emphasis on the importance of human-agent interaction and the need for a more intuitive interface challenges current consensus in the LLM/AI engineering space.
* His suggestion that artifacts will be the interface into the harness is a bold claim, as it implies a significant shift in how users interact with Cloud Code.

**Predictions, Milestones & Timelines**

* Thariq doesn't provide specific timelines for upcoming model releases or industry shifts, but mentions that Anthropic is working on evolving the harness and artifacts.
* He suggests that the next step is to create a more on-the-fly interface for harnessing the capabilities of Cloud Code.

**High-Signal Quotes**

* "I think that one of the things that's difficult as cloud code has gone broader and broader is that everyone has like their own way of using it. And it's actually very hard to like change the default behavior." - Thariq Shihipar
* "I think that artifacts are like the like way that we're trying to evolve there. But there's a lot of work to do because it's so much more complicated than like a multiple choice question." - Thariq Shihipar
* "I think that the next step is to create a more on-the-fly interface for harnessing the capabilities of Cloud Code." - Thariq Shihipar

**Actionable Takeaways for AI Engineers**

* Implement the Ask User Question tool to improve human-agent interaction and harness the capabilities of Cloud Code.
* Explore the use of artifacts to store and manage persistent data and enable multiple Clouds to access and communicate with each other.
* Consider the importance of human-agent interaction and the need for a more intuitive interface in harnessing the capabilities of Cloud Code.
* Investigate the potential for local clouds to communicate with each other seamlessly, enabling a more on-the-fly interface for harnessing the capabilities of Cloud Code.


✓ Executive briefing saved to disk: Latent_Space_Briefing_Claude_Codes_Next_Era__Thariq_Shihi.md


In [11]:
# Production Archive Engine (Auto-Append to Hugging Face Hub)
from datasets import Dataset, concatenate_datasets, load_dataset

# -------------------------------------------------------------------------
# CONFIGURATION: Enter your Hugging Face username below
# Example: HF_USERNAME = "farazahmad"
# -------------------------------------------------------------------------
HF_USERNAME = "Faraz-Ahmad"  # <-- Put your Hugging Face username between the quotes
DATASET_NAME = "latent-space-podcast-archive"
# -------------------------------------------------------------------------

if not HF_USERNAME:
    print(" Paused: Please enter your Hugging Face username in HF_USERNAME above to publish your archive.")
else:
    HF_REPO_ID = f"{HF_USERNAME}/{DATASET_NAME}"
    word_count = len(transcription.split())

    # 1. Structure the new episode as a clean data record
    new_record = [{
        "podcast": "Latent Space",
        "episode_title": episode_info["title"],
        "published_date": episode_info["published"],
        "audio_url": episode_info.get("audio_url", ""),
        "fast_test_mode": FAST_TEST_MODE,
        "word_count": word_count,
        "transcript": transcription,
        "intel_briefing": briefing_markdown,
        "processed_at": time.strftime("%Y-%m-%d %H:%M:%S")
    }]
    new_ds = Dataset.from_list(new_record)

    # 2. Check if the archive already exists on Hugging Face
    try:
        print(f"Checking for existing archive at: {HF_REPO_ID}...")
        existing_ds = load_dataset(HF_REPO_ID, split="train")
        existing_titles = set(existing_ds["episode_title"])

        # Prevent duplicate entries
        if episode_info["title"] in existing_titles:
            print(f" Notice: Episode '{episode_info['title']}' is already in the archive. Skipping duplicate row.")
            final_ds = existing_ds
        else:
            final_ds = concatenate_datasets([existing_ds, new_ds])
            print(f"✓ Appending new episode! Archive now contains {len(final_ds)} total episodes.")
    except Exception:
        # First time running: create a fresh dataset
        print(f"Creating brand-new Hugging Face dataset repository: {HF_REPO_ID}...")
        final_ds = new_ds

    # 3. Push to Hugging Face Hub
    final_ds.push_to_hub(HF_REPO_ID, token=hf_token)
    print("\n" + "=" * 65)
    print(" Archive is live on Hugging Face Hub!")
    print(f" View it here: https://huggingface.co/datasets/{HF_REPO_ID}")
    print("=" * 65)


Checking for existing archive at: Faraz-Ahmad/latent-space-podcast-archive...


README.md:   0%|          | 0.00/593 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 26.1kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpx7l93f28.parquet    : 100%|##########| 23.4kB / 23.4kB            


 Archive is live on Hugging Face Hub!
 View it here: https://huggingface.co/datasets/Faraz-Ahmad/latent-space-podcast-archive
